In [1]:
import os

# # Limit OS-level process affinity to exclusively use Core 0.
# # This strictly physically constraints Polars, PyTorch, and everything else in this notebook to 1 core.
os.sched_setaffinity(0, {1,2,3})

# Also keep your GPU and general thread limits just in case
#os.environ["CUDA_VISIBLE_DEVICES"] = "1" 
os.environ["OMP_NUM_THREADS"] = "3"
os.environ["POLARS_MAX_THREADS"] = "3" # Must be set before 'import polars'

In [5]:
print(os.sched_getaffinity(0))

{1, 2, 3}


In [ ]:
print(os.getpid())

181189


In [2]:
# Reload custom modules automatically
%load_ext autoreload
%autoreload 2
from mlp_lstm import DataPreparation, LightningLSTM, train_lstm_model

In [3]:
import torch
import lightning as L
import polars as pl
import os
import subprocess
import time
#from lightning.pytorch.loggers import TensorBoardLogger
from datetime import date, timedelta
import optuna
import logging
# We can turn off Lightning's progress bar for Optuna to keep the console clean
import warnings
warnings.filterwarnings("ignore", ".*does not have many workers.*")

print(f"PyTorch Version: {torch.__version__}")
torch.set_float32_matmul_precision('medium')

PyTorch Version: 2.5.1


In [4]:
df_tfidf = (
    pl.scan_parquet('/mnt/windows/windows_hanka_bcthesis/full_news/tfidf_nasdaq.parquet')
    # Use whatever the date column is actually called in this file
    .filter(pl.col("trading_session_date_utc").is_between(pl.date(2006, 10, 20), pl.date(2019, 12, 31)))
    .collect()
)

# Use scan_parquet() -> filter() -> collect()
df_sent = (
    pl.scan_parquet('/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2023_avg_sentiment.parquet')
    # Use whatever the date column is actually called in this file
    .filter(pl.col("trading_session_date_utc").is_between(pl.date(2006, 10, 20), pl.date(2019, 12, 31)))
    .collect()
)

df_emb = (
    pl.scan_parquet('/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2023_avg_embeddings.parquet')
    # Use whatever the date column is actually called in this file
    .filter(pl.col("trading_session_date_utc").is_between(pl.date(2006, 10, 20), pl.date(2019, 12, 31)))
    .collect()
)

# 1. Load the 5 individual stock dataframes
df_aapl = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/AAPL.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect()
df_msft = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/MSFT.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect()
df_googl = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/GOOGL.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect()
df_amzn = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/AMZN.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect()
df_nvda = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/NVDA.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect() # Or FB.csv

# Package them up
dict_of_dfs = {
    "AAPL": df_aapl.select(['date', 'close']),
    "MSFT": df_msft.select(['date', 'close']),
    "GOOGL": df_googl.select(['date', 'close']),
    "AMZN": df_amzn.select(['date', 'close']),
    "NVDA": df_nvda.select(['date', 'close'])
}

In [5]:
DataPrepObject = DataPreparation()

In [6]:
# 1. Base Layer: Prices
DataPrepObject.load_and_prepare_multiple_price_data(dict_of_dfs, start_date=date(2006, 10, 20), end_date=date(2019, 12, 31)) 
DataPrepObject.load_finbert_embeddings_data(df_emb, n_components=60)
DataPrepObject.load_tfidf_data(df_tfidf, n_components=100)
DataPrepObject.load_finbert_sentiment_data(df_sent)

Loading prices for multiple stocks: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']...
Master multi-stock timeline established. Shape: (16570, 9)
Loading FinBERT embeddings. Applying PCA to reduce to 60 dimensions...
Embeddings shape: (3570, 768)
PCA reducing to 60 components...
Embeddings PCA applied. Total cols in master: 69
Loading TF-IDF data and aligning with timeline...
TF-IDF aligned & PCA applied. Total columns in master: 169
Loading FinBERT sentiment data and aligning with timeline...
FinBERT aligned. Total columns in master: 173


trading_date_utc,ticker,target_next_day,return_lag0,return_lag1,return_lag2,return_lag3,return_lag4,return_lag5,emb_pca_0,emb_pca_1,emb_pca_2,emb_pca_3,emb_pca_4,emb_pca_5,emb_pca_6,emb_pca_7,emb_pca_8,emb_pca_9,emb_pca_10,emb_pca_11,emb_pca_12,emb_pca_13,emb_pca_14,emb_pca_15,emb_pca_16,emb_pca_17,emb_pca_18,emb_pca_19,emb_pca_20,emb_pca_21,emb_pca_22,emb_pca_23,emb_pca_24,emb_pca_25,emb_pca_26,emb_pca_27,…,pca_67,pca_68,pca_69,pca_70,pca_71,pca_72,pca_73,pca_74,pca_75,pca_76,pca_77,pca_78,pca_79,pca_80,pca_81,pca_82,pca_83,pca_84,pca_85,pca_86,pca_87,pca_88,pca_89,pca_90,pca_91,pca_92,pca_93,pca_94,pca_95,pca_96,pca_97,pca_98,pca_99,avg_pos,avg_neg,avg_neu,daily_article_count
date,str,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f64
2006-10-30,"""AAPL""",1.0,0.000124,-0.021657,0.006244,0.007773,-0.005033,0.018887,-0.408383,-1.51191,-0.553568,-0.467821,0.229624,1.282504,0.336446,0.298169,0.521651,-0.246513,0.051765,-0.367064,0.676642,-0.943535,-1.565873,0.409516,-1.291347,0.0126,0.452128,0.449509,0.783125,-0.597398,-0.01653,-0.385904,0.588087,-0.455045,0.240922,-0.957294,…,-0.033855,-0.079112,-0.081082,0.002628,0.014306,-0.202784,-0.029634,-0.002447,-0.079821,-0.018305,-0.224409,-0.034347,0.193628,0.033739,-0.157744,-0.142678,-0.011258,0.065429,-0.120254,-0.253121,0.081503,-0.012166,-0.080028,-0.167575,-0.057358,0.095902,0.017724,0.232276,-0.025057,-0.071482,0.196693,0.119336,0.153048,0.095434,0.389893,0.514862,3.0
2006-10-31,"""AAPL""",0.0,0.008207,0.000124,-0.021657,0.006244,0.007773,-0.005033,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2006-11-01,"""AAPL""",0.0,-0.02368,0.008207,0.000124,-0.021657,0.006244,0.007773,0.319845,0.182709,1.528221,0.189535,-0.774229,0.712789,-0.266697,0.082364,0.928731,-0.204755,0.463394,-0.335497,0.838686,-1.366572,-0.914557,0.183932,-0.041732,0.059874,-0.081781,0.35008,0.900571,-0.090204,0.359838,0.419868,0.095423,-0.237157,0.339387,-0.993111,…,0.03173,-0.092656,0.031237,0.006782,0.010505,-0.096164,-0.009885,-0.103919,-0.011318,-0.031042,-0.086939,0.019999,0.039038,0.090647,0.044006,-0.060085,-0.023192,0.047531,0.042766,-0.05486,0.010733,-0.051327,-0.048202,0.017715,-0.077359,0.012177,0.168586,0.076908,0.020182,-0.100997,0.102745,0.054387,0.060765,0.405802,0.040825,0.553304,3.0
2006-11-02,"""AAPL""",0.0,-0.002274,-0.02368,0.008207,0.000124,-0.021657,0.006244,-2.478588,0.766764,2.102778,2.434615,-1.38606,-0.541972,-1.325683,-1.703224,-0.60222,1.622953,-1.172049,0.089778,0.777859,0.892529,0.993805,0.704141,1.045472,0.364383,0.800517,0.320315,0.682748,0.215434,0.090245,0.80388,0.156785,-0.465599,0.941877,0.746108,…,-0.06837,-0.075385,0.071496,-0.080439,0.002369,-0.054964,0.002039,0.08533,-0.124003,-0.218448,-0.123364,0.068193,-0.131735,-0.050188,-0.083441,0.03902,-0.00243,0.02537,-0.295085,0.013242,0.054045,-0.02415,-0.128362,0.0466,0.263587,-0.027151,0.058664,-0.020062,-0.021185,0.079457,-0.093356,-0.068044,0.026826,0.792236,0.048584,0.15918,2.0
2006-11-03,"""AAPL""",1.0,-0.008736,-0.002274,-0.02368,0.008207,0.000124,-0.021657,-2.76324,-2.556471,-0.77067,-0.166453,-0.526264,1.303837,-0.260792,-0.03108,-0.734412,-0.812854,-0.363056,-0.087761,-0.434892,0.79394,0.742792,0.446526,-0.028515,0.043933,-0.584662,-0.257993,0.167012,-0.299309,0.068721,-1.39276,0.284477,-0.074972,0.124122,0.108924,…,-0.168275,0.027235,-0.034716,-0.272581,0.013566,0.05358,0.074778,0.090656,0.064447,-0.139043,-0.008661,0.047619,0.010754,-0.028931,-0.024617,0.011117,0.166411,-0.098849,-0.132171,-0.253521,0.065767,-0.082031,0.157905,0.08592,-0.019316,0.164267,-0.039446,-0.2356

In [11]:
#starting TensorBoard to visualize training logs
# Verify the log directory exists
#log_dir = '/home/sj5/Documents/hanka_bcthesis/lightning_logs'
log_dir = '/mnt/red/red_hanka_bcthesis/lightning_logs'
if os.path.exists(log_dir):
    print(f"✓ Log directory found: {log_dir}")
    # List subdirectories (each training run)
    runs = [d for d in os.listdir(log_dir) if os.path.isdir(os.path.join(log_dir, d))]
    print(f"  Available runs: {runs if runs else 'No runs yet - train a model first!'}")
else:
    print(f"✗ Log directory not found: {log_dir}")
    print("  Train a model first to create logs")

# Kill any existing TensorBoard processes
subprocess.run(['pkill', '-f', 'tensorboard'], stderr=subprocess.DEVNULL)
time.sleep(1)

# Start TensorBoard
process = subprocess.Popen([
    'tensorboard',
    '--logdir', log_dir,
    '--host', '0.0.0.0',
    '--port', '6006'
], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

time.sleep(2)
if process.poll() is not None:
    stdout, stderr = process.communicate()
    print("❌ Error starting TensorBoard:")
    print(stderr)
else:
    print("\n✓ TensorBoard started at http://localhost:6006/")
    print("  (Check VS Code PORTS tab to ensure port 6006 is forwarded)")

✓ Log directory found: /mnt/red/red_hanka_bcthesis/lightning_logs
  Available runs: ['version_26', 'lstm_runs', 'version_0', 'version_1', 'version_10', 'version_11', 'version_12', 'version_13', 'version_14', 'version_15', 'version_16', 'version_17', 'version_18', 'version_19', 'version_2', 'version_20', 'version_21', 'version_22', 'version_23', 'version_24', 'version_25', 'version_3', 'version_4', 'version_45', 'version_46', 'version_47', 'version_48', 'version_49', 'version_5', 'version_50', 'version_51', 'version_52', 'version_53', 'version_54', 'version_55', 'version_56', 'version_6', 'version_7', 'version_8', 'version_9']

✓ TensorBoard started at http://localhost:6006/
  (Check VS Code PORTS tab to ensure port 6006 is forwarded)


In [7]:
# test 5th and final Optuna optimized run (LSTM) results
storage_name = "sqlite:///optuna_universal_stock.db"
study_name = "universal_stock_v3" # Change this if your 1000-trial study had a different name
study = optuna.load_study(study_name=study_name, storage=storage_name)

# Extract all trials into a DataFrame
df_trials = study.trials_dataframe()

# Filter out PRUNED or FAILED trials, keep only COMPLETE
df_completed = df_trials[df_trials['state'] == 'COMPLETE']

# Sort by best Validation Accuracy (descending) and grab Top 10
df_optuna_lstm_top = df_completed.sort_values(by='value', ascending=False).head(10)

# Clean up the output to only show metrics and parameters
cols_to_show = ['value'] + \
               [c for c in df_optuna_lstm_top.columns if c.startswith('params_')]

# Rename 'value' to 'val_acc' for readability
df_optuna_lstm_top = df_optuna_lstm_top[cols_to_show].rename(columns={'value': 'val_acc'})

# Display nicely in Jupyter
display(df_optuna_lstm_top)

,val_acc,params_batch_size,params_dropout,params_hidden_size,params_learning_rate,params_mode,params_num_layers,params_weight_decay
338,0.580772,32,0.498514,64,0.008236,tfidf,1,0.000754
237,0.577175,32,0.537340,64,0.005830,tfidf,1,0.002171
222,0.577175,32,0.528338,64,0.006910,tfidf,1,0.001986
206,0.575213,32,0.480309,64,0.005548,tfidf,1,0.001816
234,0.573905,32,0.536068,64,0.005077,tfidf,1,0.002060
209,0.573578,32,0.502242,64,0.006193,tfidf,1,0.002087
219,0.572269,32,0.535832,64,0.007241,tfidf,1,0.001361
210,0.571942,32,0.501345,64,0.006649,tfidf,1,0.001984
386,0.571942,32,0.523572,64,0.006869,tfidf,1,0.001764
370,0.571615,32,0.524840,16,0.007252,tfidf,1,0.001225


In [9]:
# test 5th and final Optuna optimized run (LSTM) results
# select top 5 models by validation accuracy and test them
top_optuna_lstm_models = df_optuna_lstm_top.head(5)

models_to_test = []
for idx, row in top_optuna_lstm_models.iterrows():
    models_to_test.append({
        "name": f"({row['val_acc']*100:.2f}%) - nmb {idx}",
        "hidden_size": row['params_hidden_size'],
        "dropout": row['params_dropout'],
        "learning_rate": row['params_learning_rate'],
        "weight_decay": row['params_weight_decay'],
        "batch_size": row['params_batch_size'],
        "mode": row['params_mode'],
        "num_layers": row['params_num_layers']
    })

results = []

for config in models_to_test:
    mode = config["mode"]
    run_name = f"trial {config['name']} | mode={mode}"
    print(f"\n{'='*80}\nOOS TESTING: {run_name}\n{'='*80}")
    
    # Fetch Tensors and Split (Reserves the chronological end for testing)
    df_master, X_tensor, y_tensor = DataPrepObject.get_lstm_tensors(mode=mode, seq_length=5)
    train_loader, val_loader, test_loader, num_features = DataPrepObject.split_lstm_data(
        X_tensor, y_tensor, 
        train_ratio=0.60, # ~2006-2013
        val_ratio=0.15,   # ~2014-2015
        batch_size=config["batch_size"]
    )
    
    # Train the Model
    model, trainer = train_lstm_model(
    train_loader=train_loader, 
    val_loader=val_loader, 
    num_features=num_features, 
    hidden_size=config["hidden_size"],
    num_layers=config["num_layers"],
    weight_decay=config["weight_decay"],
    learning_rate=config["learning_rate"],
    dropout=config["dropout"],
    max_epochs=120,
    verbose=False   # Keep console clean
    )

    # RUN THE OUT OF SAMPLE TEST! 
    print("Running strict Out-Of-Sample evaluation on untouched data...")
    test_metrics = trainer.test(model, dataloaders=test_loader, verbose=False)[0]

    # Store the final truth
    results.append({
        "Run": run_name,
        "Test_Acc": test_metrics.get("test_acc", 0.0),
        "Test_AUC": test_metrics.get("test_auroc", 0.0),
        "Best_Val_Acc": trainer.callbacks[0].best_score.item()
    })

    # Print Leaderboard
print("\n" + "="*50)
print("FINAL OUT-OF-SAMPLE (OOS) LEADERBOARD")
print("="*50)
for r in sorted(results, key=lambda x: x["Test_Acc"], reverse=True):
    print(f"OOS Accuracy: {r['Test_Acc']:.4f} | OOS AUC: {r['Test_AUC']:.4f} | (Val Acc: {r['Best_Val_Acc']:.4f}) -> {r['Run']}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/home/sj5/.micromamba/envs/thesis_dl_2/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/sj5/Documents/linux_hanka_bcthesis/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



OOS TESTING: trial (58.08%) - nmb 338 | mode=tfidf
Applying real sliding window (Seq=5) for mode 'tfidf' with 100 features...
LSTM Tensors created. Shape: torch.Size([20390, 5, 100]) -> (Batch, Seq_Len, Features)

--- Starting Training ---


/home/sj5/.micromamba/envs/thesis_dl_2/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)
/home/sj5/.micromamba/envs/thesis_dl_2/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in false positive score
  warnings.warn(*args, **kwargs)
Metric val_acc improved. New best score: 0.522
Metric val_acc improved by 0.002 >= min_delta = 0.0. New best score: 0.524
Metric val_acc improved by 0.004 >= min_delta = 0.0. New best score: 0.528
Metric val_acc improved by 0.016 >= min_delta = 0.0. New best score: 0.544
Metric val_acc improved by 0.001 >= min_delta = 0.0. New best score: 0.545
Metric val_acc improved by 0.011 >= min_delta = 0.0. New best score: 0.557
Monitored metric val_acc did not i


Training stopped! Best Accuracy achieved: 0.5566
Running strict Out-Of-Sample evaluation on untouched data...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


...


Training stopped! Best Accuracy achieved: 0.5477
Running strict Out-Of-Sample evaluation on untouched data...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



OOS TESTING: trial (57.39%) - nmb 234 | mode=tfidf
Applying real sliding window (Seq=5) for mode 'tfidf' with 100 features...
LSTM Tensors created. Shape: torch.Size([20390, 5, 100]) -> (Batch, Seq_Len, Features)

--- Starting Training ---


Metric val_acc improved. New best score: 0.505
Metric val_acc improved by 0.053 >= min_delta = 0.0. New best score: 0.558
Monitored metric val_acc did not improve in the last 30 records. Best score: 0.558. Signaling Trainer to stop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Training stopped! Best Accuracy achieved: 0.5582
Running strict Out-Of-Sample evaluation on untouched data...

FINAL OUT-OF-SAMPLE (OOS) LEADERBOARD
OOS Accuracy: 0.5302 | OOS AUC: 0.5258 | (Val Acc: 0.5618) -> trial (57.72%) - nmb 237 | mode=tfidf
OOS Accuracy: 0.5257 | OOS AUC: 0.5217 | (Val Acc: 0.5782) -> trial (57.72%) - nmb 222 | mode=tfidf
OOS Accuracy: 0.5157 | OOS AUC: 0.5171 | (Val Acc: 0.5477) -> trial (57.52%) - nmb 206 | mode=tfidf
OOS Accuracy: 0.5141 | OOS AUC: 0.5185 | (Val Acc: 0.5582) -> trial (57.39%) - nmb 234 | mode=tfidf
OOS Accuracy: 0.5059 | OOS AUC: 0.5218 | (Val Acc: 0.5566) -> trial (58.08%) - nmb 338 | mode=tfidf


In [ ]:
# test 1st Optuna optimized run (LSTM) results
storage_name = "sqlite:///optuna_lstm.db" 
study_name = "LSTM_MegaSweep" # Change this if your 1000-trial study had a different name
study = optuna.load_study(study_name=study_name, storage=storage_name)

# Extract all trials into a DataFrame
df_trials = study.trials_dataframe()

# Filter out PRUNED or FAILED trials, keep only COMPLETE
df_completed = df_trials[df_trials['state'] == 'COMPLETE']

# Sort by best Validation Accuracy (descending) and grab Top 10
df_top = df_completed.sort_values(by='value', ascending=False).head(10)

# Clean up the output to only show metrics and parameters
cols_to_show = ['value'] + \
               [c for c in df_top.columns if c.startswith('params_')]

# Rename 'value' to 'val_acc' for readability
df_top = df_top[cols_to_show].rename(columns={'value': 'val_acc'})

# Display nicely in Jupyter
display(df_top)

,val_acc,params_dropout,params_hidden_size,params_learning_rate,params_mode,params_seq_length
900,0.642857,0.4,128,0.007604,finbert_emb,5
958,0.620130,0.4,128,0.000769,finbert_emb,5
297,0.613636,0.4,128,0.006874,finbert_emb,5
749,0.610390,0.4,128,0.001266,finbert_emb,5
185,0.607143,0.4,128,0.001270,finbert_emb,5
519,0.607143,0.4,128,0.002317,finbert_emb,5
930,0.607143,0.4,128,0.000834,finbert_emb,5
933,0.607143,0.4,128,0.000610,finbert_emb,5
787,0.607143,0.4,128,0.001692,finbert_emb,5
390,0.607143,0.4,64,0.001383,finbert_emb,5


In [ ]:
# test 2nd Optuna optimized run (LSTM) results
storage_name = "sqlite:///optuna_lstm_v2.db" 
study_name = "LSTM_DeepSweep_v2" # Change this if your 1000-trial study had a different name

study = optuna.load_study(study_name=study_name, storage=storage_name)

# Extract all trials into a DataFrame
df_trials = study.trials_dataframe()

# Filter out PRUNED or FAILED trials, keep only COMPLETE
df_completed = df_trials[df_trials['state'] == 'COMPLETE']

# Sort by best Validation Accuracy (descending) and grab Top 10
df_top = df_completed.sort_values(by='value', ascending=False).head(10)

# Clean up the output to only show metrics and parameters
cols_to_show = ['value'] + \
               [c for c in df_top.columns if c.startswith('params_')]

# Rename 'value' to 'val_acc' for readability
df_top = df_top[cols_to_show].rename(columns={'value': 'val_acc'})

# Display nicely in Jupyter
display(df_top)

,val_acc,params_batch_size,params_dropout,params_hidden_size,params_learning_rate,params_mode,params_num_layers,params_seq_length,params_weight_decay
183,0.613636,32,0.3,32,0.003462,finbert_emb,2,5,0.000014
176,0.610390,32,0.3,32,0.003452,finbert_emb,2,5,0.000008
112,0.610390,32,0.2,32,0.003125,finbert_emb,2,5,0.000011
247,0.607143,32,0.2,32,0.004820,finbert_emb,2,5,0.000008
184,0.607143,32,0.3,32,0.003405,finbert_emb,2,5,0.000015
164,0.603896,32,0.3,32,0.002761,finbert_emb,2,5,0.000009
185,0.603896,32,0.3,32,0.003626,finbert_emb,2,5,0.000015
86,0.603896,32,0.1,32,0.004458,finbert_emb,2,5,0.000011
206,0.603896,32,0.3,32,0.003921,finbert_emb,2,5,0.000006
216,0.603896,32,0.2,32,0.003310,finbert_emb,2,5,0.000009


In [ ]:
# test 3rd Optuna optimized run (LSTM) results
storage_name = "sqlite:///optuna_universal_stock.db"
study_name = "universal_stock_v1" # Change this if your 1000-trial study had a different name

study = optuna.load_study(study_name=study_name, storage=storage_name)

# Extract all trials into a DataFrame
df_trials = study.trials_dataframe()

# Filter out PRUNED or FAILED trials, keep only COMPLETE
df_completed = df_trials[df_trials['state'] == 'COMPLETE']

# Sort by best Validation Accuracy (descending) and grab Top 10
df_top = df_completed.sort_values(by='value', ascending=False).head(10)

# Clean up the output to only show metrics and parameters
cols_to_show = ['value'] + \
               [c for c in df_top.columns if c.startswith('params_')]

# Rename 'value' to 'val_acc' for readability
df_top = df_top[cols_to_show].rename(columns={'value': 'val_acc'})

# Display nicely in Jupyter
display(df_top)

,val_acc,params_batch_size,params_dropout,params_hidden_size,params_learning_rate,params_num_layers,params_weight_decay
64,0.566017,32,0.362077,32,0.005202,1,0.000021
245,0.563853,32,0.504964,64,0.007364,1,0.000222
137,0.561688,32,0.480834,32,0.005894,1,0.000176
95,0.559524,128,0.471795,32,0.002979,1,0.000147
241,0.558442,32,0.501908,64,0.006866,1,0.000265
174,0.558442,32,0.482682,128,0.006289,1,0.000063
259,0.558442,32,0.383576,64,0.004425,1,0.000475
161,0.556277,32,0.494967,128,0.005300,1,0.000048
122,0.556277,32,0.478653,128,0.004824,1,0.000101
255,0.555195,32,0.519155,64,0.004285,1,0.000414


In [ ]:
# test 4th Optuna optimized run (LSTM) results
storage_name = "sqlite:///optuna_universal_stock.db"
study_name = "universal_stock_v2" # Change this if your 1000-trial study had a different name
study = optuna.load_study(study_name=study_name, storage=storage_name)

# Extract all trials into a DataFrame
df_trials = study.trials_dataframe()

# Filter out PRUNED or FAILED trials, keep only COMPLETE
df_completed = df_trials[df_trials['state'] == 'COMPLETE']

# Sort by best Validation Accuracy (descending) and grab Top 10
df_top = df_completed.sort_values(by='value', ascending=False).head(10)

# Clean up the output to only show metrics and parameters
cols_to_show = ['value'] + \
               [c for c in df_top.columns if c.startswith('params_')]

# Rename 'value' to 'val_acc' for readability
df_top = df_top[cols_to_show].rename(columns={'value': 'val_acc'})

# Display nicely in Jupyter
display(df_top)

,val_acc,params_batch_size,params_dropout,params_hidden_size,params_learning_rate,params_mode,params_num_layers,params_weight_decay
136,0.557267,64,0.590845,32,0.000403,tfidf,3,0.000132
259,0.557267,128,0.532323,64,0.009976,tfidf_hybrid,3,0.000041
314,0.552579,128,0.575845,64,0.009053,tfidf_hybrid,3,0.000033
240,0.550569,128,0.588480,32,0.009003,tfidf_hybrid,3,0.000038
67,0.548560,64,0.461357,32,0.000102,tfidf,3,0.000036
358,0.547890,64,0.599829,32,0.009196,tfidf_hybrid,3,0.000067
356,0.547220,64,0.582929,32,0.003011,tfidf_hybrid,3,0.000059
62,0.547220,64,0.451028,32,0.006073,tfidf,3,0.000138
147,0.547220,64,0.557743,32,0.000210,tfidf_hybrid,3,0.000086
307,0.546551,32,0.572939,32,0.009060,tfidf_hybrid,3,0.000033


In [ ]:
# test 4th Optuna optimized run (LSTM) results
models_to_test = [
    {
        # The #1 overall winner from your first run. Wide network, aggressive learning rate.
        "name": "(55.72%) - nmb 136",
        "hidden_size": 32, "dropout": 0.590845, "learning_rate": 0.000403, 
        "num_layers": 3, "weight_decay": 0.000132, "batch_size": 64, "mode": "tfidf"
    },
    {
        # Trial 958 from the first run. Same wide architecture, but a much safer, traditional learning rate.
        # It's good to include this in case the 0.0076 LR above was too volatile.
        "name": "(55.72%) - nmb 259",
        "hidden_size": 64, "dropout": 0.532323, "learning_rate": 0.009976, 
        "num_layers": 3, "weight_decay": 0.000041, "batch_size": 128, "mode": "tfidf_hybrid"
    },
    {
        # The #1 winner from your deep Phase 2 sweep. Deeper network, smaller batches, smaller nodes.
        "name": "(55.25%) - nmb 314",
        "hidden_size": 64, "dropout": 0.575845, "learning_rate": 0.009053, 
        "num_layers": 3, "weight_decay": 0.000033, "batch_size": 128, "mode": "tfidf_hybrid"
    },
        {
        # The #1 winner from your deep Phase 2 sweep. Deeper network, smaller batches, smaller nodes.
        "name": "(55.05%) - nmb 240",
        "hidden_size": 32, "dropout": 0.588480, "learning_rate": 0.009003, 
        "num_layers": 3, "weight_decay": 0.000038, "batch_size": 128, "mode": "tfidf_hybrid"
    }
]

#modes = ['finbert_emb', 'finbert_emb_hybrid']
#pca_components = [60, 128, 256, 768] # Include 768 so the prep class knows to bypass PCA entirely
results = []

for config in models_to_test:
    mode = config["mode"]
    run_name = f"trial {config['name']} | mode={mode}"
    print(f"\n{'='*80}\nOOS TESTING: {run_name}\n{'='*80}")
    
    # Fetch Tensors and Split (Reserves the chronological end for testing)
    df_master, X_tensor, y_tensor = DataPrepObject.get_lstm_tensors(mode=mode, seq_length=5)
    train_loader, val_loader, test_loader, num_features = DataPrepObject.split_lstm_data(
        X_tensor, y_tensor, 
        train_ratio=0.60, # ~2006-2013
        val_ratio=0.15,   # ~2014-2015
        batch_size=config["batch_size"]
    )
    
    # Train the Model
    model, trainer = train_lstm_model(
    train_loader=train_loader, 
    val_loader=val_loader, 
    num_features=num_features, 
    hidden_size=config["hidden_size"],
    num_layers=config["num_layers"],
    weight_decay=config["weight_decay"],
    learning_rate=config["learning_rate"],
    dropout=config["dropout"],
    max_epochs=120,
    verbose=False   # Keep console clean
    )

    # RUN THE OUT OF SAMPLE TEST! 
    print("Running strict Out-Of-Sample evaluation on untouched data...")
    test_metrics = trainer.test(model, dataloaders=test_loader, verbose=False)[0]

    # Store the final truth
    results.append({
        "Run": run_name,
        "Test_Acc": test_metrics.get("test_acc", 0.0),
        "Test_AUC": test_metrics.get("test_auroc", 0.0),
        "Best_Val_Acc": trainer.callbacks[0].best_score.item()
    })

# Print Leaderboard
print("\n" + "="*50)
print("FINAL OUT-OF-SAMPLE (OOS) LEADERBOARD")
print("="*50)
for r in sorted(results, key=lambda x: x["Test_Acc"], reverse=True):
    print(f"OOS Accuracy: {r['Test_Acc']:.4f} | OOS AUC: {r['Test_AUC']:.4f} | (Val Acc: {r['Best_Val_Acc']:.4f}) -> {r['Run']}")

Loading Universal multi-stock data for MLP Optimization...
Loading prices for multiple stocks: ['AAPL', 'MSFT', 'GOOGL']...
Master multi-stock timeline established. Shape: (9942, 9)
Loading TF-IDF data and aligning with timeline...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/home/sj5/.micromamba/envs/thesis_dl_2/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/sj5/Documents/linux_hanka_bcthesis/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


TF-IDF aligned & PCA applied. Total columns in master: 109

OOS TESTING: trial (55.72%) - nmb 136 | mode=tfidf
Applying real sliding window (Seq=5) for mode 'tfidf' with 100 features...
LSTM Tensors created. Shape: torch.Size([9930, 5, 100]) -> (Batch, Seq_Len, Features)

--- Starting Training ---


Metric val_acc improved. New best score: 0.513
Metric val_acc improved by 0.028 >= min_delta = 0.0. New best score: 0.541
Metric val_acc improved by 0.011 >= min_delta = 0.0. New best score: 0.553
Monitored metric val_acc did not improve in the last 15 records. Best score: 0.553. Signaling Trainer to stop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Training stopped! Best Accuracy achieved: 0.5527
Running strict Out-Of-Sample evaluation on untouched data...


...


Training stopped! Best Accuracy achieved: 0.5218
Running strict Out-Of-Sample evaluation on untouched data...

OOS TESTING: trial (55.05%) - nmb 240 | mode=tfidf_hybrid
Applying real sliding window (Seq=5) for mode 'tfidf_hybrid' with 101 features...
LSTM Tensors created. Shape: torch.Size([9930, 5, 101]) -> (Batch, Seq_Len, Features)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



--- Starting Training ---


Metric val_acc improved. New best score: 0.506
Metric val_acc improved by 0.003 >= min_delta = 0.0. New best score: 0.508
Metric val_acc improved by 0.003 >= min_delta = 0.0. New best score: 0.511
Metric val_acc improved by 0.002 >= min_delta = 0.0. New best score: 0.513
Metric val_acc improved by 0.005 >= min_delta = 0.0. New best score: 0.518
Metric val_acc improved by 0.001 >= min_delta = 0.0. New best score: 0.520
Metric val_acc improved by 0.002 >= min_delta = 0.0. New best score: 0.522
Metric val_acc improved by 0.005 >= min_delta = 0.0. New best score: 0.527
Metric val_acc improved by 0.003 >= min_delta = 0.0. New best score: 0.530
Metric val_acc improved by 0.007 >= min_delta = 0.0. New best score: 0.537
Monitored metric val_acc did not improve in the last 15 records. Best score: 0.537. Signaling Trainer to stop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Training stopped! Best Accuracy achieved: 0.5366
Running strict Out-Of-Sample evaluation on untouched data...

FINAL OUT-OF-SAMPLE (OOS) LEADERBOARD
OOS Accuracy: 0.5159 | OOS AUC: 0.5081 | (Val Acc: 0.5218) -> trial (55.25%) - nmb 314 | mode=tfidf_hybrid
OOS Accuracy: 0.5095 | OOS AUC: 0.5052 | (Val Acc: 0.5312) -> trial (55.72%) - nmb 259 | mode=tfidf_hybrid
OOS Accuracy: 0.5079 | OOS AUC: 0.5100 | (Val Acc: 0.5366) -> trial (55.05%) - nmb 240 | mode=tfidf_hybrid
OOS Accuracy: 0.4889 | OOS AUC: 0.4978 | (Val Acc: 0.5527) -> trial (55.72%) - nmb 136 | mode=tfidf


In [ ]:
## test 1st Optuna optimized run (LSTM) results on OOS data
models_to_test = [
    {
        # The #1 overall winner from your first run. Wide network, aggressive learning rate.
        "name": "Shallow-Aggressive (64.2%)",
        "hidden_size": 128, "dropout": 0.4, "learning_rate": 0.007604, 
        "num_layers": 1, "weight_decay": 1e-4, "batch_size": 64
    },
    {
        # Trial 958 from the first run. Same wide architecture, but a much safer, traditional learning rate.
        # It's good to include this in case the 0.0076 LR above was too volatile.
        "name": "Shallow-Conservative (62.0%)",
        "hidden_size": 128, "dropout": 0.4, "learning_rate": 0.000769, 
        "num_layers": 1, "weight_decay": 1e-4, "batch_size": 64
    },
    {
        # The #1 winner from your deep Phase 2 sweep. Deeper network, smaller batches, smaller nodes.
        "name": "Deep-Narrow (61.3%)",
        "hidden_size": 32, "dropout": 0.3, "learning_rate": 0.003462, 
        "num_layers": 2, "weight_decay": 0.000014, "batch_size": 32
    }
]

modes = ['finbert_emb', 'finbert_emb_hybrid']
pca_components = [60, 128, 256, 768] # Include 768 so the prep class knows to bypass PCA entirely
results = []

for pca_val in pca_components:
    # We must reload data prep for each PCA to reset the embeddings size
    DataPrepObject = DataPreparation()
    DataPrepObject.load_and_prepare_price_data(df_prices) 
    DataPrepObject.load_finbert_embeddings_data(df_emb, n_components=pca_val) 

    for mode in modes:
        for config in models_to_test:
            run_name = f"PCA:{pca_val} | {mode} | {config['name']}"
            print(f"\n{'='*80}\nOOS TESTING: {run_name}\n{'='*80}")
            
            # Fetch Tensors and Split (Reserves the chronological end for testing)
            df_master, X_tensor, y_tensor = DataPrepObject.get_lstm_tensors(mode=mode, seq_length=5)
            train_loader, val_loader, test_loader, num_features = DataPrepObject.split_lstm_data(
                X_tensor, y_tensor, 
                train_ratio=0.60, # ~2006-2013
                val_ratio=0.15,   # ~2014-2015
                batch_size=config["batch_size"]
            )
            
            # Train the Model
            model, trainer = train_lstm_model(
                train_loader=train_loader, 
                val_loader=val_loader, 
                num_features=num_features, 
                hidden_size=config["hidden_size"],
                num_layers=config["num_layers"],
                weight_decay=config["weight_decay"],
                learning_rate=config["learning_rate"],
                dropout=config["dropout"],
                max_epochs=120,
                verbose=False   # Keep console clean
            )
            
            # RUN THE OUT OF SAMPLE TEST! 
            print("Running strict Out-Of-Sample evaluation on untouched data...")
            test_metrics = trainer.test(model, dataloaders=test_loader, verbose=False)[0]
            
            # Store the final truth
            results.append({
                "Run": run_name,
                "Test_Acc": test_metrics.get("test_acc", 0.0),
                "Test_AUC": test_metrics.get("test_auroc", 0.0),
                "Val_Acc_Achieved": trainer.callbacks[0].best_score.item()
            })

# Print Leaderboard
print("\n" + "="*50)
print("FINAL OUT-OF-SAMPLE (OOS) LEADERBOARD")
print("="*50)
for r in sorted(results, key=lambda x: x["Test_Acc"], reverse=True):
    print(f"OOS Accuracy: {r['Test_Acc']:.4f} | OOS AUC: {r['Test_AUC']:.4f} | (Val Acc: {r['Val_Acc_Achieved']:.4f}) -> {r['Run']}")

Loading prices, creating lags, and generating targets...
Price timeline established. Shape: (2811, 8)
Loading FinBERT embeddings. Applying PCA to reduce to 30 dimensions...
Embeddings shape: (2813, 768)
PCA reducing to 30 components...
Embeddings PCA applied. Total cols in master: 38
Loading prices, creating lags, and generating targets...
Price timeline established. Shape: (2811, 8)
Loading FinBERT embeddings. Applying PCA to reduce to 60 dimensions...
Embeddings shape: (2813, 768)
PCA reducing to 60 components...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/home/sj5/.micromamba/envs/thesis_dl_2/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/sj5/Documents/linux_hanka_bcthesis/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Embeddings PCA applied. Total cols in master: 68

OOS TESTING: PCA:60 | finbert_emb | Shallow-Aggressive (64.2%)
Applying real sliding window (Seq=5) for mode 'finbert_emb' with 60 features...
LSTM Tensors created. Shape: torch.Size([2809, 5, 60]) -> (Batch, Seq_Len, Features)

--- Starting Training ---


Metric val_acc improved. New best score: 0.492
Metric val_acc improved by 0.014 >= min_delta = 0.0. New best score: 0.506
Metric val_acc improved by 0.026 >= min_delta = 0.0. New best score: 0.532
Metric val_acc improved by 0.010 >= min_delta = 0.0. New best score: 0.542
Metric val_acc improved by 0.021 >= min_delta = 0.0. New best score: 0.563
Monitored metric val_acc did not improve in the last 15 records. Best score: 0.563. Signaling Trainer to stop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning


Training stopped! Best Accuracy achieved: 0.5629
Running strict Out-Of-Sample evaluation on untouched data...

OOS TESTING: PCA:60 | finbert_emb | Shallow-Conservative (62.0%)
Applying real sliding window (Seq=5) for mode 'finbert_emb' with 60 features...
LSTM Tensors created. Shape: torch.Size([2809, 5, 60]) -> (Batch, Seq_Len, Features)

--- Starting Training ---


...


Training stopped! Best Accuracy achieved: 0.5463
Running strict Out-Of-Sample evaluation on untouched data...

OOS TESTING: PCA:768 | finbert_emb_hybrid | Shallow-Conservative (62.0%)
Applying real sliding window (Seq=5) for mode 'finbert_emb_hybrid' with 769 features...
LSTM Tensors created. Shape: torch.Size([2809, 5, 769]) -> (Batch, Seq_Len, Features)

--- Starting Training ---


Metric val_acc improved. New best score: 0.508
Metric val_acc improved by 0.017 >= min_delta = 0.0. New best score: 0.525
Monitored metric val_acc did not improve in the last 15 records. Best score: 0.525. Signaling Trainer to stop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Training stopped! Best Accuracy achieved: 0.5249
Running strict Out-Of-Sample evaluation on untouched data...

OOS TESTING: PCA:768 | finbert_emb_hybrid | Deep-Narrow (61.3%)
Applying real sliding window (Seq=5) for mode 'finbert_emb_hybrid' with 769 features...
LSTM Tensors created. Shape: torch.Size([2809, 5, 769]) -> (Batch, Seq_Len, Features)

--- Starting Training ---


Metric val_acc improved. New best score: 0.556
Monitored metric val_acc did not improve in the last 15 records. Best score: 0.556. Signaling Trainer to stop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Training stopped! Best Accuracy achieved: 0.5558
Running strict Out-Of-Sample evaluation on untouched data...

FINAL OUT-OF-SAMPLE (OOS) LEADERBOARD
OOS Accuracy: 0.5334 | OOS AUC: 0.5293 | (Val Acc: 0.5582) -> PCA:60 | finbert_emb_hybrid | Deep-Narrow (61.3%)
OOS Accuracy: 0.5249 | OOS AUC: 0.5279 | (Val Acc: 0.5558) -> PCA:60 | finbert_emb | Deep-Narrow (61.3%)
OOS Accuracy: 0.5249 | OOS AUC: 0.5305 | (Val Acc: 0.5249) -> PCA:768 | finbert_emb | Shallow-Aggressive (64.2%)
OOS Accuracy: 0.5249 | OOS AUC: 0.5296 | (Val Acc: 0.5558) -> PCA:768 | finbert_emb | Shallow-Conservative (62.0%)
OOS Accuracy: 0.5220 | OOS AUC: 0.5163 | (Val Acc: 0.5511) -> PCA:768 | finbert_emb | Deep-Narrow (61.3%)
OOS Accuracy: 0.5206 | OOS AUC: 0.5250 | (Val Acc: 0.5249) -> PCA:768 | finbert_emb_hybrid | Shallow-Conservative (62.0%)
OOS Accuracy: 0.5178 | OOS AUC: 0.5175 | (Val Acc: 0.5558) -> PCA:60 | finbert_emb | Shallow-Conservative (62.0%)
OOS Accuracy: 0.5178 | OOS AUC: 0.5250 | (Val Acc: 0.5463) -> P

In [6]:
#small fake_df for testing purposes - logic seems to work really well
# 1. GENERATE FAKE DATES (15 days)
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(15)]

# 2. GENERATE FAKE PRICES
# We need enough prices to calculate returns. 
# We'll use a perfect sequence of values so returns match day numbers:
# Day 1 doesn't have a lag return (dropped)
closes = [100.0, 110.0, 121.0, 133.1, 146.41, 161.05, 177.15, 194.87, 214.36, 235.79, 259.37, 285.31, 313.84, 345.22, 379.74]
df_fake_prices = pl.DataFrame({"date": dates, "close": closes})

# 3. GENERATE FAKE SENTIMENT
df_fake_sentiment = pl.DataFrame({
    "trading_session_date_utc": dates,
    "sentiment_pos": [float(i+1) for i in range(15)],
    "sentiment_neg": [float(-(i+1)) for i in range(15)]
})

# 4. RUN THE PIPELINE
print("--- STARTING DATA PREPARATION ---")
fake_prep = DataPreparation()
fake_prep.load_and_prepare_price_data(df_fake_prices)
fake_prep.load_finbert_sentiment_data(df_fake_sentiment)

# Use sequence length of 3 so it's easy to read
df_master, X_tensor, y_tensor = fake_prep.get_lstm_tensors(mode='finbert_sent_hybrid', seq_length=3)

# 5. PRINT THE RESULTS FOR DEBUGGING
print("\n--- RAW MASTER DATAFRAME ---")
print("Notice two rows are missing: Day 1 (no previous day to calculate return) and Day 10 (no tomorrow to calculate target).")
display(df_master.head(10))

print("\n--- SLIDING WINDOW TENSORS (First 2 sequences) ---")
print(f"X_tensor Shape: {X_tensor.shape} (Batches/Sequences, Days_in_Seq, Features)")
print("Features are: [return_lag0, sentiment_pos, sentiment_neg]\n")

# Safely print up to the first 2 sequences, based on however many actually exist
num_sequences_to_print = min(2, X_tensor.shape[0])

for i in range(num_sequences_to_print):
    print(f"--- SEQUENCE {i+1} ---")
    print("X: (The 3 days of history)")
    # Print it nicely rounded so we can see our "Day numbers"
    print(torch.round(X_tensor[i] * 100) / 100) 
    print(f"y: (The target for what happens on the day AFTER the sequence ends)")
    print(f"{y_tensor[i]}\n")

--- STARTING DATA PREPARATION ---
Loading prices, creating lags, and generating targets...
Price timeline established. Shape: (8, 8)
Loading FinBERT sentiment data and aligning with timeline...
FinBERT aligned. Total columns in master: 10
Applying real sliding window (Seq=3) for mode 'finbert_sent_hybrid' with 3 features...
LSTM Tensors created. Shape: torch.Size([6, 3, 3]) -> (Batch, Seq_Len, Features)

--- RAW MASTER DATAFRAME ---
Notice two rows are missing: Day 1 (no previous day to calculate return) and Day 10 (no tomorrow to calculate target).


trading_date_utc,target_next_day,return_lag0,return_lag1,return_lag2,return_lag3,return_lag4,return_lag5,sentiment_pos,sentiment_neg
date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2024-01-07,1.0,0.099969,0.099993,0.1,0.1,0.1,0.1,7.0,-7.0
2024-01-08,1.0,0.100028,0.099969,0.099993,0.1,0.1,0.1,8.0,-8.0
2024-01-09,1.0,0.100015,0.100028,0.099969,0.099993,0.1,0.1,9.0,-9.0
2024-01-10,1.0,0.099972,0.100015,0.100028,0.099969,0.099993,0.1,10.0,-10.0
2024-01-11,1.0,0.100004,0.099972,0.100015,0.100028,0.099969,0.099993,11.0,-11.0
2024-01-12,1.0,0.100012,0.100004,0.099972,0.100015,0.100028,0.099969,12.0,-12.0
2024-01-13,1.0,0.099996,0.100012,0.100004,0.099972,0.100015,0.100028,13.0,-13.0
2024-01-14,1.0,0.099987,0.099996,0.100012,0.100004,0.099972,0.100015,14.0,-14.0



--- SLIDING WINDOW TENSORS (First 2 sequences) ---
X_tensor Shape: torch.Size([6, 3, 3]) (Batches/Sequences, Days_in_Seq, Features)
Features are: [return_lag0, sentiment_pos, sentiment_neg]

--- SEQUENCE 1 ---
X: (The 3 days of history)
tensor([[ 0.1000,  7.0000, -7.0000],
        [ 0.1000,  8.0000, -8.0000],
        [ 0.1000,  9.0000, -9.0000]])
y: (The target for what happens on the day AFTER the sequence ends)
1.0

--- SEQUENCE 2 ---
X: (The 3 days of history)
tensor([[  0.1000,   8.0000,  -8.0000],
        [  0.1000,   9.0000,  -9.0000],
        [  0.1000,  10.0000, -10.0000]])
y: (The target for what happens on the day AFTER the sequence ends)
1.0

